# Diabetes Prediction (improved)
PIMA dataset. Changes vs. the original: no data leakage (imputation + scaling live inside a Pipeline), class-balanced models, engineered features, a soft-voting ensemble, and a decision threshold tuned on out-of-fold training predictions only.

In [1]:
import numpy as np, pandas as pd, pickle, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, confusion_matrix, classification_report)

## Load data
Zeros in Glucose, BloodPressure, SkinThickness, Insulin and BMI are missing values, so they become NaN here. Imputation happens later, inside the pipeline, using training data only.

In [2]:
df = pd.read_csv('diabetes.csv')
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print(df.shape); print(df['Outcome'].value_counts())
X = df.drop(columns='Outcome'); Y = df['Outcome']

(768, 9)
Outcome
0    500
1    268
Name: count, dtype: int64


## Split first, then build the pipeline

In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

def add_features(X):
    X = X.copy()
    X["Glucose_BMI"]  = X["Glucose"] * X["BMI"]
    X["Glucose_Age"]  = X["Glucose"] * X["Age"]
    X["Age_Preg"]     = X["Age"] * X["Pregnancies"]
    X["Insulin_miss"] = X["Insulin"].isna().astype(int)       # missingness itself is informative
    X["Skin_miss"]    = X["SkinThickness"].isna().astype(int)
    return X

def make_pipeline(model):
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("impute",   SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale",    StandardScaler()),
        ("model",    model),
    ])

ensemble = VotingClassifier([
    ("rf", RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=3,
                                  class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    ("lr", LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000)),
    ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=2,
                                      subsample=0.8, random_state=0)),
], voting="soft")

best_model = make_pipeline(ensemble)

baseline = make_pipeline(RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1))

## Tune the decision threshold (training data only)
The default 0.5 cutoff under-detects diabetics. We pick the cutoff that maximizes F1 on out-of-fold predictions.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_prob = cross_val_predict(best_model, X_train, Y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

thresholds = np.arange(0.20, 0.71, 0.01)
f1_scores = [f1_score(Y_train, oof_prob >= t) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(f1_scores))])
print("Selected threshold:", round(best_threshold, 2), "| OOF F1:", round(max(f1_scores), 4))

Selected threshold: 0.34 | OOF F1: 0.6987


## Evaluate on the untouched test set

In [5]:
baseline.fit(X_train, Y_train)
best_model.fit(X_train, Y_train)

def report(name, prob, thr):
    pred = (prob >= thr).astype(int)
    print(f"{name:10s} acc {accuracy_score(Y_test, pred):.3f} | recall {recall_score(Y_test, pred):.3f} | "
          f"precision {precision_score(Y_test, pred):.3f} | F1 {f1_score(Y_test, pred):.3f} | AUC {roc_auc_score(Y_test, prob):.3f}")
    return pred

report("Baseline", baseline.predict_proba(X_test)[:, 1], 0.5)
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = report("Improved", test_prob, best_threshold)

print("\nConfusion matrix:\n", confusion_matrix(Y_test, test_pred))
print("\n", classification_report(Y_test, test_pred))
print("Train accuracy:", round(accuracy_score(Y_train, (best_model.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)), 3))

Baseline   acc 0.727 | recall 0.463 | precision 0.658 | F1 0.543 | AUC 0.822
Improved   acc 0.740 | recall 0.815 | precision 0.595 | F1 0.688 | AUC 0.829

Confusion matrix:
 [[70 30]
 [10 44]]

               precision    recall  f1-score   support

           0       0.88      0.70      0.78       100
           1       0.59      0.81      0.69        54

    accuracy                           0.74       154
   macro avg       0.73      0.76      0.73       154
weighted avg       0.78      0.74      0.75       154

Train accuracy: 0.801


## Robustness check: repeated stratified CV on the full dataset
The test set has only 154 rows, so single-split numbers are noisy. This gives a steadier comparison. Thresholds are 0.5 for the baseline and the tuned value for the improved model, and the tuned value is slightly optimistic here because it was chosen on training data.

In [6]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
for name, mdl in [("Baseline RF", baseline), ("Improved ensemble", best_model)]:
    r = cross_validate(mdl, X, Y, cv=rcv, scoring=["accuracy", "recall", "precision", "f1", "roc_auc"], n_jobs=-1)
    print(f"{name:18s} acc {r['test_accuracy'].mean():.3f} | recall {r['test_recall'].mean():.3f} | "
          f"precision {r['test_precision'].mean():.3f} | F1 {r['test_f1'].mean():.3f} | AUC {r['test_roc_auc'].mean():.3f}")

Baseline RF        acc 0.766 | recall 0.616 | precision 0.686 | F1 0.646 | AUC 0.842
Improved ensemble  acc 0.769 | recall 0.725 | precision 0.658 | F1 0.687 | AUC 0.848


## Predictive system

In [7]:
cols = list(X.columns)
input_data = (5, 166, 72, 19, 175, 25.8, 0.587, 51)
row = pd.DataFrame([input_data], columns=cols)
row[zero_as_missing] = row[zero_as_missing].replace(0, np.nan)   # same zero handling as training

prob = best_model.predict_proba(row)[0, 1]
print(f"Diabetes probability: {prob:.2f} (threshold {best_threshold:.2f})")
print('The person is diabetic' if prob >= best_threshold else 'The person is not diabetic')

Diabetes probability: 0.69 (threshold 0.34)
The person is diabetic


In [9]:
# save the pipeline AND the threshold together
with open('diabetes_model.pkl', 'wb') as f:
    pickle.dump({"model": best_model, "threshold": best_threshold, "columns": cols}, f)